# 03 — REC2026 conference paper: toy example (Figures 2–5)

Reproduction of the figures of

> C. Nardin, S. Marelli, M. Broccardo, B. Sudret. *A continuous-time Markov Chain Monte Carlo
> formulation for the PEER performance-based earthquake engineering framework.* REC 2026.

Three-state CTMC ($ds_0$ undamaged, $ds_1$ damaged, $ds_2$ collapse) with
$\lambda_{01}, \lambda_{02}, \lambda_{12} = \{0.02,\ 10^{-4},\ 0.005\}$ [year$^{-1}$],
$T_\mathrm{hor} = 50$ years and recovery rate $\mu_{10} \in \{10^{-3}, \dots, 10^{1}\}$ [year$^{-1}$].

**Two formulations of the resilience metric.** The REC2026 paper uses an *unconditional* metric,
the expected fraction of $[0, T]$ spent in $ds_0$ over all trajectories (also those that collapse):

$$\mathcal{R}_0^{\mathrm{REC}} = \frac{n_{00}(T)}{T}, \qquad n_{00}(T) = \big[(-Q_T)^{-1}(I - e^{Q_T T})\big]_{00},
\qquad \rho^{\mathrm{REC}} = -\Phi^{-1}\big(1 - \mathcal{R}_0^{\mathrm{REC}}\big).$$

The Structural Safety paper (notebooks 01–02) replaces it with a *conditional* metric: the fraction of
time in $ds_0$ **given survival**, $\mathcal{Q}_0 = 1 - \mathcal{R}_0$ (Eq. 17), with
$\rho = -\Phi^{-1}(\mathcal{R}_0)$ (Eq. 18–19). Both are computed here and compared, in closed form
and by Monte Carlo (`ctmc_pbee.montecarlo`).

| quantity | REC2026 (unconditional) | Structural Safety (conditional) |
|---|---|---|
| fraction of time in $ds_0$ | `metrics.R0_unconditional` | `metrics.R0_complement_exact` ($\mathcal{Q}_0$) |
| resilience index | `metrics.rho_unconditional` | `metrics.rho_index(metrics.R0_exact(...))` |
| Monte Carlo | `mc["R0_unconditional"]` (all trajectories) | `mc["Q0_conditional"]` (surviving trajectories) |

Figures regenerated here are written to `output/rec2026/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ctmc_pbee import core, metrics, montecarlo
from ctmc_pbee.paths import output_dir

out = output_dir('rec2026')                       # <repo>/output/rec2026

# Toy example of the REC2026 paper
lam01, lam02, lam12 = 0.02, 1e-4, 0.005           # [1/yr]
T_hor = 50.0                                      # [yr]
mu10_values = np.logspace(-3, 1, 5)               # 0.001 ... 10 [1/yr]
N_SIM = 15_000                                    # Monte Carlo trajectories per mu10
SEED = 0

col_old, col_new = 'black', '#7030A0'             # REC2026 (unconditional) / Structural Safety (conditional)
plt.rcParams.update({"mathtext.fontset": "stix", "font.size": 12})


def save(fig, name):
    fig.savefig(out / f'{name}.svg', transparent=True, format='svg', bbox_inches='tight')
    print('Saved:', out / f'{name}.svg')

## Figure 2 — Long-term state probabilities

$\pi_i(t) = \pi(0)\, e^{Qt}$ (`core.pi_t`) for a very slow recovery ($\mu_{10} = 10^{-3}$) and a moderate
one ($\mu_{10} = 10^{-1}$ year$^{-1}$).

In [ ]:
times = np.linspace(0, T_hor, 201)
fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for ax, mu, col in zip(axes, [mu10_values[0], mu10_values[2]], ['darkorange', 'darkviolet']):
    Q = core.build_Q(lam01, lam02, lam12, mu)
    P = np.array([core.pi_t(Q, t) for t in times])
    for i, ls in enumerate(['-', '--', ':']):
        ax.plot(times, P[:, i], ls, color=col, label=fr'$p(ds_{i})$')
    ax.set_title(fr'$\mu_{{10}} = {mu:.3f}$ [year$^{{-1}}$]')
    ax.set_xlabel('time [years]')
    ax.set_ylim(0, 1.05)
axes[0].set_ylabel(r'$p(ds_i)$ [-]')
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
save(fig, 'REC-fig2-long-term-probabilities')
plt.show()

## Closed form and Monte Carlo for each $\mu_{10}$

For each recovery rate: failure probability $F$ and $\beta$, the two resilience formulations, and the
Monte Carlo estimates ($N_\mathrm{sim} = 15\,000$ trajectories, fixed seed).

In [ ]:
rows, mc_runs = [], {}
for mu in mu10_values:
    Q   = core.build_Q(lam01, lam02, lam12, mu)
    Q_T = core.extract_Q_T(Q)
    F, _ = metrics.F_tau(Q_T, T_hor)
    R0_rec = metrics.R0_unconditional(Q_T, T_hor)          # REC2026
    R0_ss  = metrics.R0_exact(Q_T, T_hor)                  # Structural Safety, Eq. 18
    mc = montecarlo.run(Q, T_hor, N_SIM, seed=SEED)
    mc_runs[mu] = mc
    rows.append({
        'mu10 [1/yr]': mu,
        'F': F, 'F (MC)': mc['F'], 'beta': metrics.beta_index(F),
        'R0_REC = n00/T': R0_rec, 'R0_REC (MC)': mc['R0_unconditional'],
        'rho_REC': metrics.rho_unconditional(R0_rec),
        'Q0 = 1-R0 (SS)': 1 - R0_ss, 'Q0 (MC)': mc['Q0_conditional'],
        'rho_SS': metrics.rho_index(R0_ss),
    })
res = pd.DataFrame(rows)
res.round(4)

## Figure 3 — $F$ and fraction of time in $ds_0$ vs $\mu_{10}$: closed form vs Monte Carlo

(a) failure probability (Eq. 7 of the REC2026 paper, `metrics.F_tau`); (b) fraction of time in $ds_0$:
unconditional $\mathcal{R}_0^{\mathrm{REC}}$ (black, as published) and conditional $\mathcal{Q}_0$ (violet,
Structural Safety). Circles: closed form; triangles: Monte Carlo. The shaded band marks the $\mu_{10}$
used in Figure 4.

In [ ]:
mu_conv = mu10_values[1]                       # mu10 of the convergence study (Figure 4)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))

ax = axes[0]
ax.plot(res['mu10 [1/yr]'], res['F'], '-', color='gray', lw=0.8)
ax.scatter(res['mu10 [1/yr]'], res['F'], s=80, color='black', label='closed form')
ax.scatter(res['mu10 [1/yr]'], res['F (MC)'], s=80, marker='^', facecolor='white',
           edgecolor='black', label='Monte Carlo')
ax.set_ylabel(r'$\mathcal{F}(\tau_c)$ [-]')

ax = axes[1]
for key, key_mc, col, lab in [('R0_REC = n00/T', 'R0_REC (MC)', col_old, r'$\mathcal{R}_0^{REC} = n_{00}/T$'),
                              ('Q0 = 1-R0 (SS)', 'Q0 (MC)', col_new, r'$\mathcal{Q}_0 = 1-\mathcal{R}_0$ (SS)')]:
    ax.plot(res['mu10 [1/yr]'], res[key], '-', color=col, lw=0.8, alpha=0.6)
    ax.scatter(res['mu10 [1/yr]'], res[key], s=80, color=col, label=lab)
    ax.scatter(res['mu10 [1/yr]'], res[key_mc], s=80, marker='^', facecolor='white', edgecolor=col)
ax.set_ylabel(r'fraction of time in $ds_0$ [-]')
ax.set_ylim(0.5, 1.05)

for ax in axes:
    ax.set_xscale('log')
    ax.set_xlabel(r'$\mu_{10}$ [years$^{-1}$]')
    ax.axvspan(mu_conv / 1.4, mu_conv * 1.4, color='#c8c8f0', alpha=0.6, zorder=0)
    ax.grid(True, ls='--', alpha=0.5)
    ax.legend(fontsize=9)
plt.tight_layout()
save(fig, 'REC-fig3-ftau-r0')
plt.show()

## Figure 4 — Monte Carlo convergence ($\mu_{10} = 10^{-2}$ year$^{-1}$)

Running mean of the Monte Carlo estimators with their 95% confidence band, against the closed form
(dashed): (a) $\mathcal{F}$, (b) $\mathcal{R}_0^{\mathrm{REC}}$ (all trajectories), (c) $\mathcal{Q}_0$
(surviving trajectories only; the x-axis counts surviving trajectories).

In [ ]:
mc = mc_runs[mu_conv]
r = res.loc[res['mu10 [1/yr]'] == mu_conv].iloc[0]
surv = ~mc['collapsed']
series = [
    (mc['F_samples'],           r['F'],              r'$\mathcal{F}(\tau_c)$',       (0.0, 0.2)),
    (mc['R0_samples'],          r['R0_REC = n00/T'], r'$\mathcal{R}_0^{REC}$',       (0.6, 0.8)),
    (mc['R0_samples'][surv],    r['Q0 = 1-R0 (SS)'], r'$\mathcal{Q}_0$ (SS)',         (0.6, 0.8)),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for ax, (samples, ref, lab, ylim) in zip(axes, series):
    n, mean, lo, hi = montecarlo.running_mean_ci(samples)
    k = 100                                         # skip the first, very noisy estimates
    ax.plot(n[k:], mean[k:], color='tab:blue', label=lab + ', MC')
    ax.fill_between(n[k:], lo[k:], hi[k:], color='blue', alpha=0.15, label='95% CI')
    ax.axhline(ref, ls='--', color='tab:blue', label=lab + ', closed form')
    ax.set_ylim(ylim)
    ax.set_xlabel(r'$N_\mathrm{sim}$ [-]')
    ax.legend(fontsize=9)
plt.tight_layout()
save(fig, 'REC-fig4-montecarlo-convergence')
plt.show()

## Figure 5 — $(\beta, \rho)$ domain for varying $\mu_{10}$

Grey circles: $\rho^{\mathrm{REC}}$ (as published, marker size increasing with $\mu_{10}$); violet: $\rho$ of
the Structural Safety paper. $\beta$ is the same in both.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.axvspan(2, 4, color='green', alpha=0.10, zorder=0)
ax.axhspan(2, 4, color='green', alpha=0.10, zorder=0)
sizes = np.linspace(80, 400, len(res))
ax.scatter(res['beta'], res['rho_REC'], s=sizes, color='#555555', edgecolor='black',
           linewidth=1.5, zorder=5, label=r'$\rho^{REC} = -\Phi^{-1}(1-n_{00}/T)$')
ax.scatter(res['beta'], res['rho_SS'], s=sizes, facecolor='none', edgecolor=col_new,
           linewidth=1.8, zorder=6, label=r'$\rho = -\Phi^{-1}(\mathcal{R}_0)$ (SS)')
for _, row in res.iterrows():
    ax.annotate(fr"$\mu_{{10}}={row['mu10 [1/yr]']:g}$", (row['beta'], row['rho_REC']),
                textcoords='offset points', xytext=(12, -4), fontsize=9)
ax.set_xlim(0, 4); ax.set_ylim(0, 4)
ax.set_xlabel(r'Reliability index, $\beta$')
ax.set_ylabel(r'Resilience index, $\rho$')
ax.grid(True, ls='--', alpha=0.5)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
save(fig, 'REC-fig5-beta-rho-domain')
plt.show()